# Starbucks US Expansion: 30 Years of State-Level Growth (Animated Choropleth)

*How 1,004 stores became 16,935 — mapped by state, normalized by population, animated by year*

**Series context:** This is Notebook D in the Starbucks data science series. See also:
- [Manhattan Cafe Wars](https://www.kaggle.com/code/shiratoriseto/manhattan-cafe-wars-starbucks-vs-1200-competitors) — EDA & competitor mapping (Theme 0)
- [Starbucks 10-K NLP](https://www.kaggle.com/code/shiratoriseto/starbucks-10-k-nlp-topic-keyword-analysis) — 30 years of corporate language (Theme 1)
- [Spatial Clustering](https://www.kaggle.com/code/shiratoriseto/starbucks-spatial-clustering) — Moran's I, LISA, Ripley's K (Theme 2A)
- [Location Fitness](https://www.kaggle.com/code/shiratoriseto/starbucks-location-fitness) — Demand-supply scoring (Theme 2B)
- [Data Pipeline](https://www.kaggle.com/code/shiratoriseto/starbucks-data-pipeline-edgar-osm-to-csv) — Full reproducibility (Notebook C)
- [Walk-Distance Analysis](https://www.kaggle.com/code/shiratoriseto/starbucks-walk-distance-analysis-osmnx) — OSMnx walk-distance analysis (Theme 2C)
- [LDA Topic Explorer](https://www.kaggle.com/code/shiratoriseto/starbucks-10-k-lda-topic-explorer-pyldavis) — Interactive LDA exploration (Theme 1F)

---

**What you'll learn:**
1. **State-level store estimation** — combining 10-K national totals with a single geographic snapshot
2. **Population-normalized density** — stores per 100K reveals saturation, not just raw count
3. **Animated choropleth** — 30-year spatial expansion in one interactive visualization
4. **Bar chart race** — US vs International growth trajectories

**The punchline:**
> *Washington state had 21 stores per 100K residents by 2025 — 3× the national average. Meanwhile, Mississippi has just 1.7 per 100K.*


## Section 0 — Setup & Data Loading

We combine three data sources:
1. **10-K store counts** (FY1996–2025) — US national totals extracted from SEC filings
2. **2017 Kaggle snapshot** (directory.csv) — state-level store distribution for 13,608 US locations
3. **Census population** (1996–2025) — state-level population for density normalization


In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# ----- Data path auto-detect -----
KAGGLE_PATHS = [
    Path("/kaggle/input/starbucks-30year-nlp-corpus"),
    Path("/kaggle/input/datasets/shiratoriseto/starbucks-30year-nlp-corpus"),
]
KAGGLE_PATHS_DIR = [
    Path("/kaggle/input/store-locations"),
    Path("/kaggle/input/datasets/starbucks/store-locations"),
    Path("/kaggle/input/datasets/organizations/starbucks/store-locations"),
]
LOCAL_NLP = Path("../../data/processed/sec-edgar")
LOCAL_RAW = Path("../../data/raw/kaggle_starbucks")
LOCAL_EXT = Path("../../data/external")

# Find NLP corpus dataset
DATA_DIR = None
for p in KAGGLE_PATHS:
    if p.exists():
        DATA_DIR = p
        break
if DATA_DIR is None and LOCAL_NLP.exists():
    DATA_DIR = LOCAL_NLP

# Find directory.csv (2017 snapshot)
DIR_CSV = None
for p in KAGGLE_PATHS_DIR:
    candidate = p / "directory.csv"
    if candidate.exists():
        DIR_CSV = candidate
        break
if DIR_CSV is None and (LOCAL_RAW / "directory.csv").exists():
    DIR_CSV = LOCAL_RAW / "directory.csv"

# Find state population
POP_CSV = None
for p in KAGGLE_PATHS:
    candidate = p / "state_population_1996_2025.csv"
    if candidate.exists():
        POP_CSV = candidate
        break
if POP_CSV is None and (LOCAL_EXT / "state_population_1996_2025.csv").exists():
    POP_CSV = LOCAL_EXT / "state_population_1996_2025.csv"

print(f"NLP corpus: {DATA_DIR}")
print(f"Directory CSV: {DIR_CSV}")
print(f"Population CSV: {POP_CSV}")

# Load data
timeseries = pd.read_csv(DATA_DIR / "store_counts_timeseries.csv")
print(f"\nTimeseries: FY{timeseries['fiscal_year'].min()}–FY{timeseries['fiscal_year'].max()} ({len(timeseries)} years)")
print(f"  US total range: {timeseries['us_total'].min():,} – {timeseries['us_total'].max():,}")

if DIR_CSV is not None:
    directory = pd.read_csv(DIR_CSV)
    us_stores = directory[directory['Country'] == 'US'].copy()
    print(f"\n2017 snapshot: {len(us_stores):,} US stores across {us_stores['State/Province'].nunique()} states")
else:
    print("\n⚠ directory.csv not found. Using hardcoded 2017 state shares.")
    us_stores = None

state_pop = pd.read_csv(POP_CSV)
print(f"\nPopulation data: {state_pop['state'].nunique()} states × {state_pop['year'].nunique()} years")


## Section 1 — From One Snapshot to 30 Years: The Constant-Share Method

We have only **one geographic snapshot** (2017) but **30 years of national totals**. To estimate state-level counts for every year, we assume each state's share of US stores is roughly constant over time.

$$\text{stores}_{\text{state, year}} = \text{US\_total}_{\text{year}} \times \text{share}_{\text{state, 2017}}$$

**Is this valid?** Reasonably so for 2005–2025 (Starbucks was in all 50 states by 2000, and state shares stabilized). For 1996–2004, the assumption is weaker — early Starbucks was concentrated on the West Coast. We flag this uncertainty visually.


In [ ]:
# ----- Compute state shares from 2017 snapshot -----
if us_stores is not None:
    state_counts_2017 = us_stores.groupby('State/Province').size().reset_index(name='count_2017')
else:
    # Fallback: hardcoded top state shares
    state_counts_2017 = pd.DataFrame({
        'State/Province': ['CA','TX','WA','FL','NY','IL','AZ','CO','VA','OH','PA','NJ','GA','NC','MA',
                           'OR','MN','MD','MI','IN','TN','MO','WI','CT','NV','SC','KY','UT','AL','LA',
                           'OK','KS','DC','IA','NE','AR','HI','NM','NH','ID','MS','RI','ME','DE','MT',
                           'ND','SD','WV','WY','AK','VT'],
        'count_2017': [2821,1042,757,694,645,575,488,481,432,378,373,369,356,319,311,
                       293,240,233,220,193,188,173,170,156,146,143,115,115,105,104,
                       96,88,91,85,64,60,57,53,50,48,35,33,32,24,22,
                       18,17,16,14,13,12]
    })

state_counts_2017['share'] = state_counts_2017['count_2017'] / state_counts_2017['count_2017'].sum()
state_counts_2017 = state_counts_2017.sort_values('share', ascending=False).reset_index(drop=True)

# Verify against 10-K FY2017 total
fy2017_10k = timeseries.loc[timeseries['fiscal_year'] == 2017, 'us_total'].iloc[0]
snapshot_total = state_counts_2017['count_2017'].sum()
print(f"2017 comparison: snapshot = {snapshot_total:,}, 10-K = {fy2017_10k:,}, diff = {fy2017_10k - snapshot_total:+,} ({(fy2017_10k - snapshot_total)/fy2017_10k:.1%})")

# Show top 10
print(f"\nTop 10 states by 2017 store share:")
for _, row in state_counts_2017.head(10).iterrows():
    print(f"  {row['State/Province']:>2s}: {row['count_2017']:>5,} stores ({row['share']:.1%})")

print(f"\nTotal states: {len(state_counts_2017)}")
print(f"Top 5 states hold: {state_counts_2017.head(5)['share'].sum():.1%} of all US stores")


## Section 2 — 30 Years × 51 States: Building the Expansion Matrix

Apply the constant-share assumption to create a complete state × year matrix, then normalize by population.


In [ ]:
# ----- Build state × year matrix -----
years = timeseries['fiscal_year'].values
us_totals = timeseries.set_index('fiscal_year')['us_total']

rows = []
for _, state_row in state_counts_2017.iterrows():
    st = state_row['State/Province']
    share = state_row['share']
    for yr in years:
        estimated = round(us_totals[yr] * share)
        rows.append({'state': st, 'year': yr, 'estimated_stores': estimated, 'share': share})

state_year = pd.DataFrame(rows)

# Merge population
state_year = state_year.merge(state_pop, on=['state', 'year'], how='left')

# Stores per 100K population
state_year['stores_per_100k'] = state_year['estimated_stores'] / state_year['population'] * 100_000

# Sanity checks
print(f"Matrix: {state_year['state'].nunique()} states × {state_year['year'].nunique()} years = {len(state_year):,} rows")
print(f"Population coverage: {state_year['population'].notna().mean():.0%}")

# Show 2025 top 10 by density
latest = state_year[state_year['year'] == 2025].sort_values('stores_per_100k', ascending=False)
print(f"\n2025 — Top 10 states by stores per 100K:")
for _, row in latest.head(10).iterrows():
    print(f"  {row['state']:>2s}: {row['estimated_stores']:>5,.0f} stores, {row['stores_per_100k']:.1f} per 100K")

print(f"\n2025 — Bottom 5:")
for _, row in latest.tail(5).iterrows():
    print(f"  {row['state']:>2s}: {row['estimated_stores']:>5,.0f} stores, {row['stores_per_100k']:.1f} per 100K")

# National average
nat_avg = state_year[state_year['year'] == 2025]
total_stores = nat_avg['estimated_stores'].sum()
total_pop = nat_avg['population'].sum()
print(f"\nNational average (2025): {total_stores/total_pop*100000:.1f} per 100K")


## Section 3 — The Main Event: 30-Year Animated Choropleth

Press play to watch Starbucks spread across America. The color scale shows **stores per 100K residents** — population-normalized density that reveals saturation, not just raw store count.

**What to watch for:**
- **1996–2000:** Intense West Coast concentration (WA/OR/CA glow bright)
- **2000–2008:** Rapid nationwide expansion — the map fills in
- **2008–2010:** Post-crisis contraction (colors dim slightly)
- **2010–2025:** Gradual recovery with density plateaus in mature markets

> ⚠ **Confidence note:** Store estimates for 1996–2004 use the constant-share assumption, which overstates non-West-Coast states in early years. The dashed line in Fig. 2 marks the lower-confidence period.


In [ ]:
# ----- Fig. 1: Animated choropleth — stores per 100K by state -----
# Filter to states with population data
choropleth_data = state_year.dropna(subset=['stores_per_100k']).copy()

fig1 = px.choropleth(
    choropleth_data,
    locations='state',
    locationmode='USA-states',
    color='stores_per_100k',
    animation_frame='year',
    scope='usa',
    color_continuous_scale='YlOrRd',
    range_color=[0, choropleth_data['stores_per_100k'].quantile(0.95)],
    labels={'stores_per_100k': 'Stores per 100K', 'state': 'State', 'year': 'Year'},
    title='Fig. 1 — Starbucks Store Density by State (1996–2025)',
    hover_data={'estimated_stores': ':,.0f', 'population': ':,.0f'},
)

fig1.update_layout(
    geo=dict(
        showlakes=True, lakecolor='rgb(255,255,255)',
        showframe=False,
    ),
    height=550, width=900,
    coloraxis_colorbar=dict(title='Stores<br>per 100K'),
    title_x=0.5,
)

# Slow down animation
fig1.layout.updatemenus[0].buttons[0].args[1]['frame']['duration'] = 800
fig1.layout.updatemenus[0].buttons[0].args[1]['transition']['duration'] = 300

fig1.show()


## Section 4 — Growth Trajectories: Which States Grew Fastest?


In [ ]:
# ----- Fig. 2: Line chart — stores per 100K over time for select states -----
highlight_states = ['WA', 'CA', 'CO', 'NY', 'TX', 'FL', 'MS']
colors = {'WA': '#D62828', 'CA': '#F77F00', 'CO': '#457B9D', 'NY': '#2A9D8F',
          'TX': '#E76F51', 'FL': '#264653', 'MS': '#8D99AE'}

fig2 = go.Figure()

# Background: all other states in light grey
for st in state_year['state'].unique():
    if st not in highlight_states:
        st_data = state_year[state_year['state'] == st].sort_values('year')
        fig2.add_trace(go.Scatter(
            x=st_data['year'], y=st_data['stores_per_100k'],
            mode='lines', line=dict(color='#ddd', width=0.5),
            showlegend=False, hoverinfo='skip',
        ))

# Highlighted states
for st in highlight_states:
    st_data = state_year[state_year['state'] == st].sort_values('year')
    fig2.add_trace(go.Scatter(
        x=st_data['year'], y=st_data['stores_per_100k'],
        mode='lines+markers', name=st,
        line=dict(color=colors[st], width=2.5),
        marker=dict(size=3),
    ))

# Confidence boundary
fig2.add_vrect(x0=1996, x1=2004, fillcolor='grey', opacity=0.08,
               annotation_text='Lower confidence<br>(constant-share assumption)',
               annotation_position='top left', annotation_font_size=9)

fig2.update_layout(
    title='Fig. 2 — Store Density Trajectories by State (1996–2025)',
    xaxis_title='Fiscal Year', yaxis_title='Stores per 100K Residents',
    height=450, width=900, template='plotly_white',
    legend=dict(x=0.02, y=0.98),
)
fig2.show()

# Growth stats
print("Growth: 1996 vs 2025 (stores per 100K):")
for st in highlight_states:
    st_data = state_year[state_year['state'] == st].sort_values('year')
    d1996 = st_data[st_data['year'] == 1996]['stores_per_100k'].iloc[0]
    d2025 = st_data[st_data['year'] == 2025]['stores_per_100k'].iloc[0]
    print(f"  {st}: {d1996:.1f} → {d2025:.1f} ({d2025/d1996:.1f}×)")


## Section 5 — Bar Chart Race: US vs International


In [ ]:
# ----- Fig. 3: Animated bar chart — US vs International, CO vs Licensed -----
bar_data = timeseries[['fiscal_year', 'co_us', 'lic_us', 'co_international', 'lic_international']].copy()
bar_data = bar_data.melt(id_vars='fiscal_year', var_name='segment', value_name='stores')

# Clean labels
label_map = {
    'co_us': 'US Company-Operated',
    'lic_us': 'US Licensed',
    'co_international': 'Intl Company-Operated',
    'lic_international': 'Intl Licensed',
}
bar_data['segment'] = bar_data['segment'].map(label_map)

color_map = {
    'US Company-Operated': '#00704A',
    'US Licensed': '#1E3932',
    'Intl Company-Operated': '#D4E9E2',
    'Intl Licensed': '#F2F0EB',
}

fig3 = px.bar(
    bar_data,
    x='fiscal_year', y='stores',
    color='segment',
    color_discrete_map=color_map,
    labels={'fiscal_year': 'Fiscal Year', 'stores': 'Number of Stores', 'segment': 'Segment'},
    title='Fig. 3 — US vs International: The Strategic Pivot (1996–2025)',
    height=450, width=900,
)

fig3.update_layout(
    barmode='stack',
    template='plotly_white',
    legend=dict(x=0.02, y=0.98, bgcolor='rgba(255,255,255,0.8)'),
    xaxis=dict(dtick=2),
)
fig3.show()

# Key metrics
us_2025 = timeseries.loc[timeseries['fiscal_year'] == 2025, 'us_total'].iloc[0]
intl_2025 = timeseries.loc[timeseries['fiscal_year'] == 2025, 'intl_total'].iloc[0]
total_2025 = timeseries.loc[timeseries['fiscal_year'] == 2025, 'total_worldwide'].iloc[0]
pct_intl = intl_2025 / total_2025 * 100

us_1996 = timeseries.loc[timeseries['fiscal_year'] == 1996, 'us_total'].iloc[0]
intl_1996 = timeseries.loc[timeseries['fiscal_year'] == 1996, 'intl_total'].iloc[0]

print(f"\n1996: US = {us_1996:,}, International = {intl_1996:,} ({intl_1996/max(us_1996+intl_1996,1):.0%} intl)")
print(f"2025: US = {us_2025:,}, International = {intl_2025:,} ({pct_intl:.0%} intl)")
print(f"\nUS grew {us_2025/us_1996:.1f}× while International grew from {intl_1996:,} to {intl_2025:,}")


## Section 6 — Saturation Analysis: Where Is Starbucks Running Out of Room?


In [ ]:
# ----- Fig. 4: Saturation scatter — 2025 density vs 2010-2025 growth -----
# Growth rate = (density_2025 - density_2010) / density_2010
pivot_density = state_year.pivot_table(index='state', columns='year', values='stores_per_100k')

growth_data = pd.DataFrame({
    'state': pivot_density.index,
    'density_2010': pivot_density[2010].values,
    'density_2025': pivot_density[2025].values,
})
growth_data['growth_pct'] = (growth_data['density_2025'] - growth_data['density_2010']) / growth_data['density_2010'] * 100
growth_data = growth_data.dropna()

fig4 = px.scatter(
    growth_data,
    x='density_2010', y='growth_pct',
    text='state',
    labels={'density_2010': 'Store Density 2010 (per 100K)',
            'growth_pct': 'Density Growth 2010→2025 (%)'},
    title='Fig. 4 — Saturation vs Growth: Already-Dense States Grow Slower',
    height=450, width=700,
    template='plotly_white',
)
fig4.update_traces(textposition='top center', marker=dict(size=8, color='#00704A'))
fig4.show()

r = growth_data['density_2010'].corr(growth_data['growth_pct'])
print(f"Correlation: 2010 density vs 2010-2025 growth = {r:.3f}")
print(f"→ {'Negative' if r < 0 else 'Positive'} correlation — {'denser states grow slower (saturation effect)' if r < 0 else 'no saturation yet'}")

# Top/bottom growers
print(f"\nFastest growing (density increase):")
for _, row in growth_data.nlargest(5, 'growth_pct').iterrows():
    print(f"  {row['state']}: {row['density_2010']:.1f} → {row['density_2025']:.1f} (+{row['growth_pct']:.0f}%)")
print(f"\nSlowest growing:")
for _, row in growth_data.nsmallest(5, 'growth_pct').iterrows():
    print(f"  {row['state']}: {row['density_2010']:.1f} → {row['density_2025']:.1f} ({row['growth_pct']:+.0f}%)")


## Section 7 — Limitations & Methodology Notes

### Key Assumptions

| Assumption | Impact | Mitigation |
|---|---|---|
| **Constant state share** | Overestimates non-West-Coast states before ~2004 | Grey confidence band on Fig. 2; noted in all visualizations |
| **Single calibration point (2017)** | Cannot detect share shifts over time | If a 2023 snapshot becomes available, re-calibrate with two anchor points |
| **Population extrapolation (2022–2025)** | Growth rates assumed constant from 2020–2021 trend | Impact is <1% on density calculations |
| **10-K totals include temporary closures** | Some years include stores that opened and closed within the year | Net count is still the best available metric |

### Data Sources

| Source | Coverage | License |
|---|---|---|
| Starbucks 10-K filings (SEC EDGAR) | FY1996–2025 national totals | Public domain (gov filing) |
| Kaggle directory.csv (2017) | 13,608 US stores by state | CC0 |
| US Census Bureau (PEP/ACS) | State population 1996–2025 | Public domain |

### The Series

| Notebook | Topic | Link |
|----------|-------|------|
| Theme 0 | Manhattan Cafe Wars — EDA | [Link](https://www.kaggle.com/code/shiratoriseto/manhattan-cafe-wars-starbucks-vs-1200-competitors) |
| Theme 1 | 10-K NLP Analysis | [Link](https://www.kaggle.com/code/shiratoriseto/starbucks-10-k-nlp-topic-keyword-analysis) |
| Theme 2A | Spatial Clustering | [Link](https://www.kaggle.com/code/shiratoriseto/starbucks-spatial-clustering) |
| Theme 2B | Location Fitness | [Link](https://www.kaggle.com/code/shiratoriseto/starbucks-location-fitness) |
| **Notebook D** | **US Expansion Animation** | **You are here** |
| Notebook C | Data Pipeline | [Link](https://www.kaggle.com/code/shiratoriseto/starbucks-data-pipeline-edgar-osm-to-csv) |

---

*Built with pandas · plotly · US Census Bureau API*
